In [1]:
# LOAD DATA

In [2]:
import torch 
import torch.nn as nn
import torch.optim as optim 

import torchvision
from torchvision.datasets import MNIST

In [3]:
# DATASET AND DATALOADER 

import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(), #scale and tensor
    transforms.Normalize((0.5,),(0.5,)) #1 channel == 1 val for std dev and mean
])

train_set = MNIST(root="./data" , train=True , download="True" , transform = transform)
test_set = MNIST(root="./data" , train=False , download="True" , transform = transform)

In [4]:
train_set
test_set

Dataset MNIST
    Number of datapoints: 10000
    Root location: ./data
    Split: Test
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5,), std=(0.5,))
           )

In [5]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_set , batch_size = 64 , shuffle= True)
test_loader = DataLoader(test_set , batch_size = 64 , shuffle= True)

### BUILD CNN


In [6]:
# https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html

In [28]:
class CNN (nn.Module):
    def __init__(self) :
        super(CNN , self).__init__()

        self.conv_layers = nn.Sequential(

            # (28,28,1)
            # 1st layer
            nn.Conv2d(1, 28 ,kernel_size = 3, padding=1) ,#in_chanel , output channel , filter, padding
            nn.ReLU() ,  #(28,28,28),
            nn.MaxPool2d(3, stride=2),  #kernel = 3 
                #(14,14,28)

            # 2nd layer
            nn.Conv2d(28, 56 ,kernel_size = 3, padding=1) ,
            nn.ReLU()  , #(14,14,56),
            nn.MaxPool2d(3, stride=2), #(7,7,56)

            # 3rd layer
            nn.Conv2d(56, 112 ,kernel_size = 3, padding=1) ,
            nn.ReLU() ,  #(7,7,112)
            nn.MaxPool2d(3, stride=2) ,#(3,3,112) == (2,2,112)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(2*2*112 , 224 ) ,#input , output 
            nn.ReLU(),

                 # output -- softmax applied byt cross entropy
            nn.Linear(224 , 10), #10 output = 0-9digits
        )

    def forward(self , x):
        x = self.conv_layers(x)
        x = x.view(x.size(0) , -1) #flattening x
        x = self.fc_layers(x) 

        return x

In [29]:
model = CNN()

criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

## Train CNN

In [30]:
epochs = 10 

for epoch in range(epochs):
    epoch_train_loss= 0.0 

    for images , labels in train_loader :
        model.train()
        optimizer.zero_grad() 
        
        output = model.forward(images) 
        loss = criteria(output , labels) 
        loss.backward() 
        optimizer.step() 

        epoch_train_loss += loss.item()
        
    print(f"epoch={epoch+1}/{epochs} and loss = {epoch_train_loss/len(train_loader)}")

epoch=1/10 and loss = 0.19963719324667506
epoch=2/10 and loss = 0.0479043358486386
epoch=3/10 and loss = 0.03548416873437168
epoch=4/10 and loss = 0.028552183637680705
epoch=5/10 and loss = 0.0241787523417419
epoch=6/10 and loss = 0.019830977385994597
epoch=7/10 and loss = 0.017925072739893102
epoch=8/10 and loss = 0.0155552187612995
epoch=9/10 and loss = 0.015222834281727911
epoch=10/10 and loss = 0.010944524966333136


## Evaluate

In [32]:
model.eval()
total_labels = 0 
correct_labels = 0 

with torch.no_grad():
    for images, labels  in test_loader:
        outputs = model.forward(images) 
        max , predicted = torch.max(outputs , 1)

        correct_labels += (predicted ==  labels).sum().item()
        total_labels += labels.size(0)

print(f"Accuracy = {correct_labels/total_labels *100}")

Accuracy = 99.28
